# Modeling Memory in Time-Respecting Paths on Temporal Networks
### End-to-End Demonstration Tutorial

This tutorial demonstrates the end-to-end workflow implementing the methodology from:
> **Guerrini, Cattuto, Dall'Amico**, *'Modeling memory in time-respecting paths on temporal networks'*, **EPJ Data Science** (2026).

#### Outline
1. **Load Proximity Contacts & Construct Temporal Graph**
2. **Sample Non-Backtracking Time-Respecting Paths (TRPs)**
3. **Extract Transition Records & Fit MEM Memory Curves ($p$ vs. $m$)**
4. **Fit Community-Aware MEM+SBM Model & Visualize Affinity Matrix $C$**
5. **Simulate SIR Epidemic Spreading & Targeted Isolation Interventions**

In [ ]:
import sys
from pathlib import Path

# Add package source directories to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))
sys.path.insert(0, str(Path.cwd() / 'src'))
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from memory_trp.loading import load_contacts
from memory_trp.aggregation import split_by_day, build_temporal_graph
from memory_trp.trp import generate_many_trps
from memory_trp.records import build_records
from memory_trp.models import fit_mem_fixed_point, fit_mem_sbm
from memory_trp.nulls import er_null_graph
from memory_trp.sir import simulate_sir, calibrate_rate, get_highest_contact_nodes
from config import T_RES_SECONDS, RANDOM_SEED

print('Environment and modules loaded successfully!')

## 1. Load Contacts & Build Temporal Graph
We load the **Primary School** dataset (students and teachers tracked in 20-second proximity snapshots) and construct the Day 1 temporal graph.

In [ ]:
contacts, labels = load_contacts('primaryschool')
days = split_by_day(contacts, 'primaryschool')
n = len(labels)

# Construct temporal graph for Day 1
graph = build_temporal_graph(days[0], n=n, labels=labels, t_res=T_RES_SECONDS, name='PrimarySchool_Day1')
print(f'Nodes: {graph.n}')
print(f'Communities (Classes + Teachers): {len(np.unique(labels))}')
print(f'Snapshots in Day 1: {len(graph.snapshots):,}')

## 2. Sample Non-Backtracking Time-Respecting Paths (TRPs)
We sample random walks where transitions respect the temporal arrow of time ($t_{a+1} \ge t_a$) and do not backtrack ($v_{a+1} \ne v_{a-1}$).

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
num_paths = 1500
path_length = 30

paths = generate_many_trps(graph, num_paths=num_paths, length=path_length, rng=rng)
print(f'Successfully generated {len(paths):,} valid TRPs.')
print('Sample path (node sequence):', paths[0][0][:10])
print('Sample path (snapshot timestamps):', paths[0][1][:10])

## 3. Extract Transition Records & Fit MEM Memory Curves
For horizons $m \in [1, 2, 3, 5, 7, 10, 15]$, we extract transition records and estimate the memory parameter $p$ on the real graph vs. an Erdős–Rényi static null.

In [ ]:
# Generate Erdős–Rényi static null benchmark
graph_null = er_null_graph(graph, rng=rng)
paths_null = generate_many_trps(graph_null, num_paths=num_paths, length=path_length, rng=rng)

m_values = [1, 2, 3, 5, 7, 10, 15]
p_real = []
p_null = []

for m in m_values:
    rec_real = build_records(paths, m=m)
    rec_null = build_records(paths_null, m=m)
    
    fit_r = fit_mem_fixed_point(rec_real, n=n)
    fit_n = fit_mem_fixed_point(rec_null, n=n)
    
    p_real.append(fit_r.p)
    p_null.append(fit_n.p)

# Plot memory curves
plt.figure(figsize=(7, 4.5), dpi=150)
plt.plot(m_values, p_real, marker='o', linewidth=2.0, color='#1f77b4', label='Empirical Network (MEM)')
plt.plot(m_values, p_null, marker='s', linestyle='--', linewidth=1.8, color='#ff7f0e', label='Erdős–Rényi Null')
plt.xlabel('Memory horizon $m$', fontsize=11)
plt.ylabel('Inferred memory $p$', fontsize=11)
plt.title('Memory Parameter $p$ vs. Look-back Horizon $m$', fontsize=12, fontweight='bold')
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(frameon=True)
plt.ylim(-0.02, max(p_real) * 1.25)
plt.show()

## 4. Community-Aware Memory: MEM+SBM Fitting
We fit the joint MEM+SBM model to extract both the memory parameter $p$ and the normalized community affinity matrix $C$ across school classes.

In [ ]:
rec_m10 = build_records(paths, m=10)
sbm_fit = fit_mem_sbm(rec_m10, labels=labels)

print(f'MEM+SBM Fitted memory: p = {sbm_fit.p:.4f}')
print(f'Log-Likelihood: {sbm_fit.loglik:.1f}')
print(f'BIC: {sbm_fit.bic:.1f}')

# Visualize community affinity matrix C
plt.figure(figsize=(6, 5), dpi=150)
plt.imshow(sbm_fit.C, cmap='YlGnBu', origin='upper')
plt.colorbar(label='Community Affinity $C_{cd}$')
plt.title('Fitted MEM+SBM Affinity Matrix $C$', fontsize=12, fontweight='bold')
plt.xlabel('Class Group Index', fontsize=10)
plt.ylabel('Class Group Index', fontsize=10)
plt.show()

## 5. Epidemic Dynamics: SIR Spreading & 10% Targeted Isolation
We calibrate a critical transmission rate $\beta$ and simulate an SIR epidemic under:
1. **Baseline**: No intervention.
2. **Targeted Isolation**: Removing the top 10% highest-contact individuals from the start.

In [ ]:
# 1. Calibrate transmission rate to critical percolation regime (~40%)
beta = calibrate_rate(graph, target_final_fraction=0.40, recovery_snapshots=30, n_calib_runs=20, rng=rng)
print(f'Calibrated transmission rate beta: {beta:.4f}')

# 2. Select top 10% highest-contact nodes
k_nodes = int(round(0.10 * n))
contact_nodes = get_highest_contact_nodes(graph, budget=k_nodes)

# 3. Run Monte Carlo simulations
n_sims = 50
outbreaks_none = [simulate_sir(graph, beta, recovery_snapshots=30, rng=rng).final_fraction for _ in range(n_sims)]
outbreaks_isolated = [simulate_sir(graph, beta, recovery_snapshots=30, rng=rng, removed_nodes=contact_nodes).final_fraction for _ in range(n_sims)]

mean_none, std_none = float(np.mean(outbreaks_none)), float(np.std(outbreaks_none))
mean_iso, std_iso = float(np.mean(outbreaks_isolated)), float(np.std(outbreaks_isolated))
reduction = (mean_none - mean_iso) / mean_none * 100.0

print(f'Baseline Outbreak (None):         {mean_none*100:.1f}% (+/- {std_none*100:.1f}%)')
print(f'Top 10% Contact Time Isolation:   {mean_iso*100:.1f}% (+/- {std_iso*100:.1f}%)')
print(f'Relative Reduction in Outbreak:   {reduction:.1f}%')

# 4. Plot comparative bar chart
plt.figure(figsize=(5.5, 4.5), dpi=150)
bars = plt.bar(['No Intervention', 'Top 10% Isolated'], [mean_none, mean_iso],
               yerr=[std_none, std_iso], capsize=5, color=['#d62728', '#1f77b4'], edgecolor='black', linewidth=0.8)
plt.ylabel('Mean Final Outbreak Size (Fraction)', fontsize=11)
plt.title('SIR Epidemic: 10% Targeted Intervention', fontsize=12, fontweight='bold')
plt.grid(True, axis='y', linestyle=':', alpha=0.6)
plt.ylim(0, max(mean_none + std_none, mean_iso + std_iso) + 0.1)
plt.show()